## Autenticación EDL mediante earthaccess

En este tutorial demostramos cómo heredar credenciales de autenticación EDL desde [earthaccess](https://earthaccess.readthedocs.io/en/stable/) y usarlas con PyDAP.

Hay dos formas de usar `earthaccess` para autenticarse:

1. Recuperar credenciales y almacenarlas localmente en un archivo `.netrc`.
2. Heredar un objeto de sesión de requests desde `earthaccess` con el encabezado de token EDL, y usarlo para crear un objeto de sesión PyDAP optimizado (que puede ser una [CachedSession](https://requests-cache.readthedocs.io/en/stable/)).


Aunque no es estrictamente necesario, [earthaccess](https://earthaccess.readthedocs.io/en/stable/) oculta la abstracción necesaria para autenticarse mediante usuario/contraseña o token para Earthdata Login.


In [ ]:
import earthaccess
from pydap.client import create_session

### 1. Almacenar credenciales de autenticación EDL para reutilizarlas después

Esto se logra en una sola línea.

```{note}
La siguiente línea asume que tienes un archivo netrc en tu sistema. También puedes usar `strategy="interactive"`.
```


In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) 

Las credenciales se han almacenado en un archivo `.netrc`. Puedes crear un objeto de sesión de requests y este las recuperará automáticamente.


### 2. Heredar un token EDL

Esto se logra recuperando un objeto request.session inicializado por earthaccess. Contendrá el encabezado del token EDL, y puede usarse para crear un objeto `CachedSession` que habilita metadatos consolidados persistentes en flujos de trabajo con pydap.


In [ ]:
my_session = create_session(session=auth.get_session())
my_session

In [ ]:
cache_session = create_session(use_cache=True, session=auth.get_session())
cache_session

Esta `cache_session` es un objeto [requests_cache.CachedSession](https://requests-cache.readthedocs.io/en/stable/) que se puede usar con PyDAP. Internamente, PyDAP solo habilitará caché de metadatos (es decir, el `.dmr` de los archivos remotos) y los datos de dimensiones (arreglos 1D). Acelera significativamente la agregación del Dataset a nivel de colección cuando se usa `Xarray`, al usar `pydap.client.consolidate_metadata`.


### Resumen

Puedes usar el objeto `session` o `cache_session` para transmitir datos. Cada uno te permitirá autenticarte mediante usuario/contraseña o mediante token, respectivamente.
